# 用户调研 9.12 数据治理复核
沿用原有 participant_exclusions 和历史试运行判断；与 9.10 导出及清洗名单核对。
已在项目 privacy-display/.venv/bin/python 环境执行通过。从项目或其子目录运行；重跑会覆盖本目录派生 CSV 和 JSON，不修改原始数据。


In [1]:
from pathlib import Path
import collections, csv, hashlib, io, json, math, sys

ROOT = Path.cwd()
if not (ROOT / 'privacy-display/webstudy/analyze_study.py').exists():
    ROOT = next(p for p in ROOT.parents if (p / 'privacy-display/webstudy/analyze_study.py').exists())
sys.path.insert(0, str(ROOT / 'privacy-display/webstudy'))
import analyze_study as rules
SRC = ROOT / '用户调研9.12'
OLD = ROOT / '用户调研9.10'
OUT = SRC / 'cleaned'
OUT.mkdir(exist_ok=True)
data = json.loads((SRC / 'data.json').read_text())
previous = json.loads((OLD / 'data.json').read_text())
participants = sorted(data['participants'], key=lambda r: r['participant_id'])
by_pid = {p['participant_id']: p for p in participants}
old_ids = {p['participant_id'] for p in previous['participants']}
pilot_ids = {1, 2, 3, 4, 5}  # 沿用旧报告已确认的历史试运行；先验证旧记录未改变。
checks = {}
for section, key in [('participants', 'participant_id'), ('typing', 'id'), ('ratings', 'id')]:
    old_map = {r[key]: r for r in previous[section]}
    new_map = {r[key]: r for r in data[section]}
    assert len(new_map) == len(data[section]), f'{section}: duplicate key'
    assert all(k in new_map and new_map[k] == r for k, r in old_map.items())
    checks[f'{section}_unchanged_old_rows'] = len(old_map)
for key in ['participant_id', 'session_uuid', 'student_id', 'name', 'registration_index']:
    counts = collections.Counter(str(p[key]).strip() for p in participants if p[key] not in (None, ''))
    checks[f'duplicate_{key}_excess'] = sum(n - 1 for n in counts.values())
    assert checks[f'duplicate_{key}_excess'] == 0

typing, ratings = collections.defaultdict(list), collections.defaultdict(list)
for section, grouped in [('typing', typing), ('ratings', ratings)]:
    for row in data[section]:
        assert row['participant_id'] in by_pid
        grouped[row['participant_id']].append(dict(row, mask_meta_json=json.dumps(row['mask_meta'], ensure_ascii=False)))
    for rows in grouped.values():
        rows.sort(key=lambda r: r['trial_index'] if section == 'typing' else r['order_index'])

# CSV 每个导出字段均与 JSON 的参与者信息及试次信息进行对照。
with (SRC / 'privacy_display_study.csv').open(encoding='utf-8-sig', newline='') as f:
    csv_rows = list(csv.DictReader(f))
event_maps = {
    'typing': {(r['participant_id'], r['trial_index']): r for r in data['typing']},
    'rating': {(r['participant_id'], r['order_index']): r for r in data['ratings']},
}
assert len(event_maps['typing']) == len(data['typing'])
assert len(event_maps['rating']) == len(data['ratings'])
seen = set()
for row in csv_rows:
    kind, pid = row['row_type'], int(row['participant_id'])
    index = int(row['trial_index'] if kind == 'typing' else row['order_index'])
    assert (kind, pid, index) not in seen
    seen.add((kind, pid, index))
    event = event_maps[kind][(pid, index)]
    expected = {**by_pid[pid], **event, 'mask_meta_json': event['mask_meta']}
    for field, actual in row.items():
        value = expected.get(field)
        if field in ['screen_json', 'mask_meta_json']:
            value = json.loads(value) if isinstance(value, str) else value
            ok = json.loads(actual) == value
        elif value is None:
            ok = actual == ''
        elif isinstance(value, (float, int)):
            ok = actual != '' and math.isclose(float(actual), float(value), rel_tol=1e-12, abs_tol=1e-12)
        else:
            ok = actual == str(value)
        assert ok, f'CSV/JSON mismatch: {kind}, pid={pid}, index={index}, field={field}'
assert len(csv_rows) == len(data['typing']) + len(data['ratings'])
checks['csv_json_matching_event_rows'] = len(csv_rows)
checks['csv_json_fields_per_row'] = len(csv_rows[0])
checks['invalid_accuracy_values'] = sum(not 0 <= r['accuracy'] <= 1 for r in data['typing'])
checks['invalid_rating_values'] = sum(not 1 <= r[d] <= 5 for r in data['ratings'] for d in rules.RATING_DIMENSIONS)
checks['missing_consent_or_screening'] = sum(not p['consent_confirmed'] or not p['photosensitivity_screen_passed'] for p in participants)
assert checks['invalid_accuracy_values'] == checks['invalid_rating_values'] == checks['missing_consent_or_screening'] == 0

labels = {'pilot_session': '历史试运行（沿用旧报告）', 'rating_straightline_minimum_view': '6个条件均观看≤11秒，且每个条件内三项评分相同', 'control_accuracy_below_50pct': 'control两次平均准确率低于50%'}
audit = []
for p in participants:
    pid = p['participant_id']
    reasons = rules.participant_exclusions(p, typing[pid], ratings[pid])
    if pid in pilot_ids:
        reasons = ['pilot_session'] + reasons
    rs = ratings[pid]
    audit.append({'participant_id': pid, 'participant_code': '', 'batch': 'historical' if pid in old_ids else 'new',
        'date_utc': p['started_at'][:10], 'status': 'excluded' if reasons else 'included',
        'reasons': ';'.join(reasons), 'reason_zh': '；'.join(labels.get(r, r) for r in reasons),
        'typing_rows': len(typing[pid]), 'rating_rows': len(rs),
        'control_mean_accuracy': rules.mean([r['accuracy'] for r in typing[pid] if r['condition'] == 'control']),
        'minimum_rating_view_ms': min(r['view_duration_ms'] for r in rs),
        'maximum_rating_view_ms': max(r['view_duration_ms'] for r in rs),
        'within_condition_straightline_rows': sum(rules.is_straightline_rating_row(r) for r in rs)})
included = [r for r in audit if r['status'] == 'included']
excluded = [r for r in audit if r['status'] == 'excluded']
code_map = {r['participant_id']: f'P{i:03d}' for i, r in enumerate(included, 1)}
for r in audit:
    r['participant_code'] = code_map.get(r['participant_id'], '')
# 与旧清洗名单核对，确保历史样本纳入结果一致。
with (OLD / 'cleaned/cleaned_typing_trials.csv').open() as f:
    old_included = {int(r['participant_id']) for r in csv.DictReader(f)}
assert {r['participant_id'] for r in included if r['batch'] == 'historical'} == old_included

def write_csv(name, rows, fields=None):
    with (OUT / name).open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields or list(rows[0]), extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

write_csv('participant_audit.csv', audit)
write_csv('excluded_participants.csv', excluded)
# 复用旧脱敏导出的字段结构和按 pid 排序的 P 编号，历史 P001–P094 保持不变。
for name, grouped in [('cleaned_typing_trials.csv', typing), ('cleaned_rating_trials.csv', ratings), ('included_participants.csv', None)]:
    with (OLD / 'cleaned' / name).open() as f:
        fields = next(csv.reader(f))
    output_rows = []
    for pid, code in code_map.items():
        source_rows = grouped[pid] if grouped is not None else [by_pid[pid]]
        output_rows.extend({**r, 'participant_code': code} for r in source_rows)
    write_csv(name, output_rows, fields)
    assert not {'name', 'student_id', 'session_uuid'} & set(fields)

summary = {
    'raw_sessions': len(participants), 'historical_sessions': len(old_ids),
    'new_sessions': len(participants) - len(old_ids), 'pilot_sessions': len(pilot_ids),
    'formal_sessions': len(participants) - len(pilot_ids),
    'included_sessions': len(included), 'excluded_sessions_including_pilots': len(excluded),
    'formal_quality_exclusions': sum(r['participant_id'] not in pilot_ids for r in excluded),
    'new_included': sum(r['batch'] == 'new' for r in included),
    'new_excluded': sum(r['batch'] == 'new' for r in excluded),
    'included_typing_rows': sum(r['typing_rows'] for r in included),
    'included_rating_rows': sum(r['rating_rows'] for r in included),
    'excluded_typing_rows': sum(r['typing_rows'] for r in excluded),
    'excluded_rating_rows': sum(r['rating_rows'] for r in excluded),
    'exclusions': {str(r['participant_id']): r['reasons'] for r in excluded},
    'by_date_utc': {day: {'total': sum(r['date_utc'] == day for r in audit), 'included': sum(r['date_utc'] == day for r in included), 'excluded': sum(r['date_utc'] == day for r in excluded)} for day in sorted({r['date_utc'] for r in audit})},
    'checks': checks,
    'sha256': {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in [SRC / 'data.json', SRC / 'privacy_display_study.csv', ROOT / 'privacy-display/webstudy/analyze_study.py']},
}
assert len(included) + len(excluded) == len(participants)
assert all(r['typing_rows'] == 4 and r['rating_rows'] == 6 for r in audit)
(OUT / 'governance_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n')
print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "raw_sessions": 117,
  "historical_sessions": 105,
  "new_sessions": 12,
  "pilot_sessions": 5,
  "formal_sessions": 112,
  "included_sessions": 106,
  "excluded_sessions_including_pilots": 11,
  "formal_quality_exclusions": 6,
  "new_included": 12,
  "new_excluded": 0,
  "included_typing_rows": 424,
  "included_rating_rows": 636,
  "excluded_typing_rows": 44,
  "excluded_rating_rows": 66,
  "exclusions": {
    "1": "pilot_session",
    "2": "pilot_session",
    "3": "pilot_session",
    "4": "pilot_session",
    "5": "pilot_session",
    "25": "rating_straightline_minimum_view",
    "33": "rating_straightline_minimum_view",
    "38": "rating_straightline_minimum_view",
    "40": "control_accuracy_below_50pct",
    "48": "control_accuracy_below_50pct",
    "70": "rating_straightline_minimum_view"
  },
  "by_date_utc": {
    "2026-07-03": {
      "total": 1,
      "included": 0,
      "excluded": 1
    },
    "2026-07-06": {
      "total": 4,
      "included": 0,
      "excluded": 4